<a href="https://www.kaggle.com/code/akramlaamari/toon239d687cce?scriptVersionId=282102774" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# 1. Install critical libraries and update pip
!pip install -U pip -q

# 2. Fix the specific conflicts causing your error
# We pin protobuf to 3.20.x to fix the 'MessageFactory' error
# We pin pyarrow to <15.0 to fix the Kaggle cudf conflict
!pip install "protobuf<=3.20.3" "pyarrow<15.0.0" -q

# 3. Install your required libraries
!pip install toml transformers[torch] datasets accelerate -q

import pandas as pd
import numpy as np
import os
import toml
import torch
import sys
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoConfig,
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    EarlyStoppingCallback
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 14.0.2 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires pyarrow>=15.0.2, but you have pyarrow 14.0.2 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompa

2025-11-27 04:43:05.183379: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764218585.360374      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764218585.408370      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# ############################################################### THe SOlid WAY
# # PHASE 1: ETL RIGOUREUX (Anti-Fuite de Données)
# #
# # 1. Chargement & Nettoyage
# # 2. Mapping (Super-Classes)
# # 3. SPLIT (Train/Val/Test) --> CRUCIAL : AVANT L'ÉQUILIBRAGE
# # 4. Équilibrage (Uniquement sur le TRAIN)
# # 5. Conversion & Sauvegarde
# ###############################################################

# import pandas as pd
# import numpy as np
# import os
# import toml
# from sklearn.model_selection import train_test_split
# from sklearn.utils import resample
# from tqdm.auto import tqdm  # Bibliothèque pour la barre de progression

# print("===== Démarrage Phase 1 (ETL Sécurisé & Barre de Progression) =====")

# # --- 1. CONFIGURATION ---
# # ⚠️ ADAPTER SELON VOTRE ENVIRONNEMENT (KAGGLE ou COLAB)
# # Exemple Kaggle :
# DATA_DIR = "/kaggle/input/cc20218/CSE-CIC-IDS2018/"
# SAVE_DIR = "/kaggle/working/Toonset/"

# # Exemple Colab (décommenter si besoin) :
# # DATA_DIR = "/content/drive/MyDrive/LLM/CSE-CIC-IDS2018/CSE-CIC-IDS2018/"
# # SAVE_DIR = "/content/drive/MyDrive/LLM/Toon_Full_Dataset/"

# TRAIN_FILE = os.path.join(SAVE_DIR, 'train_full.toon')
# VALIDATION_FILE = os.path.join(SAVE_DIR, 'validation_full.toon')
# TEST_FILE = os.path.join(SAVE_DIR, 'test_full.toon')

# os.makedirs(SAVE_DIR, exist_ok=True)

# # Liste des fichiers
# ALL_FILES = [
#     "Bot.csv", "Brute Force -Web.csv", "Brute Force -XSS.csv",
#     "DDOS attack-HOIC.csv", "DDOS attack-LOIC-UDP.csv", "DDoS attacks-LOIC-HTTP.csv",
#     "DoS attacks-GoldenEye.csv", "DoS attacks-Hulk.csv", "DoS attacks-SlowHTTPTest.csv",
#     "DoS attacks-Slowloris.csv", "FTP-BruteForce.csv", "Infilteration.csv",
#     "SQL Injection.csv", "SSH-Bruteforce.csv"
# ]

# # Mapping des Classes (7 Super-Classes)
# LABEL_MAPPING = {
#     "Benign": "Benign",
#     "Bot": "Bot",
#     "DDOS attack-HOIC": "DDoS", "DDOS attack-LOIC-UDP": "DDoS", "DDoS attacks-LOIC-HTTP": "DDoS",
#     "DoS attacks-GoldenEye": "DoS", "DoS attacks-Hulk": "DoS", "DoS attacks-SlowHTTPTest": "DoS", "DoS attacks-Slowloris": "DoS",
#     "FTP-BruteForce": "BruteForce", "SSH-Bruteforce": "BruteForce",
#     "Brute Force -Web": "Web", "Brute Force -XSS": "Web", "SQL Injection": "Web",
#     "Infilteration": "Infiltration"
# }

# MIN_SAMPLES_TRAIN = 5000  # Seuil pour l'oversampling du TRAIN uniquement

# # --- 2. FONCTIONS DE TRAITEMENT ---

# def clean_and_map_df(df):
#     """Nettoie, renomme les colonnes et applique le mapping."""
#     # Nettoyage valeurs
#     df.replace([np.inf, -np.inf], np.nan, inplace=True)
#     df.fillna(0, inplace=True)
    
#     # Standardisation colonnes
#     df.columns = [c.strip().replace(' ', '_').replace('/', '_').upper() for c in df.columns]

#     # Renommage Label
#     if 'LABEL' not in df.columns:
#         for col in df.columns:
#             if "LABEL" in col:
#                 df.rename(columns={col: 'LABEL'}, inplace=True)
#                 break
    
#     if 'LABEL' not in df.columns: return None, None

#     # Suppression métadonnées
#     cols_to_drop = ['FLOW_ID', 'SRC_IP', 'SRC_PORT', 'DST_IP', 'TIMESTAMP', 'DATE']
#     df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True, errors='ignore')

#     # Mapping Super-Classes
#     df['LABEL'] = df['LABEL'].map(LABEL_MAPPING).fillna("Other")
#     df = df[df['LABEL'] != "Other"] # On retire les labels inconnus

#     features = [col for col in df.columns if col != 'LABEL']
#     return df, features

# def create_toon_entries(df, features):
#     """Convertit un DataFrame en liste de dictionnaires TOON rapidement."""
#     # Utilisation de zip pour aller très vite sans apply()
#     # On convertit tout en string d'un coup
#     input_data = df[features].astype(str).values.tolist()
#     target_data = df['LABEL'].astype(str).tolist()
    
#     entries = []
#     # Join avec "|"
#     for row, target in zip(input_data, target_data):
#         entries.append({
#             "input": "|".join(row),
#             "target": target
#         })
#     return entries

# def balance_training_data(df):
#     """Logique d'équilibrage appliquée UNIQUEMENT au Train set."""
#     balanced_chunks = []
#     unique_labels = df['LABEL'].unique()
    
#     print(f"   ... Équilibrage des classes : {unique_labels}")
    
#     for label in unique_labels:
#         sub = df[df['LABEL'] == label]
#         count = len(sub)
        
#         if label == "Benign":
#             # Undersampling Benign (20%)
#             sub = sub.sample(frac=0.2, random_state=42)
#         elif count < MIN_SAMPLES_TRAIN:
#             # Oversampling Rare (xN jusqu'à 5000)
#             sub = resample(sub, replace=True, n_samples=MIN_SAMPLES_TRAIN, random_state=42)
        
#         balanced_chunks.append(sub)
        
#     return pd.concat(balanced_chunks)

# # --- 3. EXÉCUTION PRINCIPALE ---

# def run_phase_1():
#     # 1. Chargement global
#     all_dfs = []
#     final_features = []
    
#     print("--- 1. Lecture des Fichiers ---")
#     # TQDM pour la barre de progression des fichiers
#     for filename in tqdm(ALL_FILES, desc="Loading CSVs"):
#         path = os.path.join(DATA_DIR, filename)
#         if not os.path.exists(path): continue
        
#         try:
#             # Lecture partielle si mémoire limitée (sinon tout lire)
#             df = pd.read_csv(path, low_memory=False)
#             df, feats = clean_and_map_df(df)
            
#             if df is not None:
#                 all_dfs.append(df)
#                 if not final_features: final_features = feats
                
#         except Exception as e:
#             print(f"Erreur {filename}: {e}")

#     if not all_dfs:
#         print("Erreur: Aucune donnée chargée.")
#         return

#     # Fusion
#     full_df = pd.concat(all_dfs, ignore_index=True)
#     print(f"Total Données Brutes : {len(full_df)} lignes")
    
#     # 2. SPLIT STRICT (Avant équilibrage !)
#     # On sépare d'abord le TEST (20%) qui doit rester 'Puir' (Distribution réelle)
#     print("--- 2. Split Train/Test (Stratifié) ---")
#     train_val_df, test_df = train_test_split(full_df, test_size=0.2, stratify=full_df['LABEL'], random_state=42)
    
#     # On sépare ensuite Train et Validation
#     train_df, val_df = train_test_split(train_val_df, test_size=0.2, stratify=train_val_df['LABEL'], random_state=42)
    
#     # Libération mémoire
#     del full_df, train_val_df, all_dfs
    
#     # 3. ÉQUILIBRAGE (Uniquement sur TRAIN)
#     print("--- 3. Équilibrage (Train uniquement) ---")
#     train_df_balanced = balance_training_data(train_df)
    
#     print(f"   -> Train (Balanced): {len(train_df_balanced)} | Val (Pure): {len(val_df)} | Test (Pure): {len(test_df)}")

#     # 4. CONVERSION ET SAUVEGARDE
#     print("--- 4. Conversion et Sauvegarde ---")
    
#     datasets = [
#         ("TRAIN", train_df_balanced, TRAIN_FILE),
#         ("VALIDATION", val_df, VALIDATION_FILE),
#         ("TEST", test_df, TEST_FILE)
#     ]
    
#     for name, df, filepath in datasets:
#         print(f"   Traitement {name}...")
#         # Conversion rapide
#         entries = create_toon_entries(df, final_features)
        
#         # Sauvegarde
#         with open(filepath, 'w', encoding='utf-8') as f:
#             toml.dump({"entry": entries}, f)
#         print(f"   ✅ {name} sauvegardé dans {filepath}")

# if __name__ == "__main__":
#     # run_phase_1()

In [3]:
# ###############################################################
# # PHASE 1: ETL "EXPERT AGENT" (Sémantique + Log-Norm + Split)
# #
# # Améliorations :
# # 1. SPLIT D'ABORD : Garantit aucune fuite de données.
# # 2. LOG-NORMALIZATION : Compresse les grands nombres (Bytes/Duration).
# # 3. SEMANTIC PREFIXES : Ajoute "dp_", "dur_" pour que T5 comprenne le sens.
# ###############################################################

# import pandas as pd
# import numpy as np
# import os
# import toml
# from sklearn.model_selection import train_test_split
# from sklearn.utils import resample
# from tqdm.auto import tqdm

# print("===== Démarrage Phase 1 (Mode Expert : Sémantique + Log) =====")

# # --- 1. CONFIGURATION ---
# DATA_DIR = "/kaggle/input/cc20218/CSE-CIC-IDS2018/"
# SAVE_DIR = "/kaggle/working/Toonset/"

# TRAIN_FILE = os.path.join(SAVE_DIR, 'train_full.toon')
# VALIDATION_FILE = os.path.join(SAVE_DIR, 'validation_full.toon')
# TEST_FILE = os.path.join(SAVE_DIR, 'test_full.toon')

# os.makedirs(SAVE_DIR, exist_ok=True)

# ALL_FILES = [
#     "Bot.csv", "Brute Force -Web.csv", "Brute Force -XSS.csv",
#     "DDOS attack-HOIC.csv", "DDOS attack-LOIC-UDP.csv", "DDoS attacks-LOIC-HTTP.csv",
#     "DoS attacks-GoldenEye.csv", "DoS attacks-Hulk.csv", "DoS attacks-SlowHTTPTest.csv",
#     "DoS attacks-Slowloris.csv", "FTP-BruteForce.csv", "Infilteration.csv",
#     "SQL Injection.csv", "SSH-Bruteforce.csv"
# ]

# LABEL_MAPPING = {
#     "Benign": "Benign",
#     "Bot": "Bot",
#     "DDOS attack-HOIC": "DDoS", "DDOS attack-LOIC-UDP": "DDoS", "DDoS attacks-LOIC-HTTP": "DDoS",
#     "DoS attacks-GoldenEye": "DoS", "DoS attacks-Hulk": "DoS", "DoS attacks-SlowHTTPTest": "DoS", "DoS attacks-Slowloris": "DoS",
#     "FTP-BruteForce": "BruteForce", "SSH-Bruteforce": "BruteForce",
#     "Brute Force -Web": "Web", "Brute Force -XSS": "Web", "SQL Injection": "Web",
#     "Infilteration": "Infiltration"
# }

# MIN_SAMPLES_TRAIN = 5000 

# # --- 2. FONCTIONS INTELLIGENTES ---

# def clean_and_map_df(df):
#     """Nettoyage + Mapping + Log-Normalization (Math pour T5)"""
#     df.replace([np.inf, -np.inf], np.nan, inplace=True)
#     df.fillna(0, inplace=True)
#     df.columns = [c.strip().replace(' ', '_').replace('/', '_').upper() for c in df.columns]

#     if 'LABEL' not in df.columns:
#         for col in df.columns:
#             if "LABEL" in col:
#                 df.rename(columns={col: 'LABEL'}, inplace=True)
#                 break
#     if 'LABEL' not in df.columns: return None, None

#     cols_to_drop = ['FLOW_ID', 'SRC_IP', 'SRC_PORT', 'DST_IP', 'TIMESTAMP', 'DATE']
#     df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True, errors='ignore')

#     # Mapping Classes
#     df['LABEL'] = df['LABEL'].map(LABEL_MAPPING).fillna("Other")
#     df = df[df['LABEL'] != "Other"]

#     # --- EXPERT TRICK 1 : LOG NORMALIZATION ---
#     # T5 ne comprend pas la différence entre 1000 et 1000000.
#     # On utilise log1p pour réduire l'échelle (0 à ~20).
#     # Cela aide pour : Duration, Bytes, Packets
#     cols_to_log = [c for c in df.columns if any(x in c for x in ['BYTES', 'DURATION', 'LENGTH', 'PKTS', 'IAT'])]
    
#     for c in cols_to_log:
#         # On s'assure que c'est numérique
#         if pd.api.types.is_numeric_dtype(df[c]):
#             df[c] = np.log1p(df[c].clip(lower=0)).round(2)

#     features = [col for col in df.columns if col != 'LABEL']
#     return df, features

# def create_toon_entries(df, features):
#     """Conversion avec INJECTION D'INDICATEURS SÉMANTIQUES (Prefixes)"""
    
#     # Prétraitement des noms de colonnes pour créer des préfixes courts
#     # Ex: DST_PORT -> dp_, FLOW_DURATION -> dur_
#     col_prefixes = []
#     for col in features:
#         if "DST_PORT" in col: p = "dp_"      # Destination Port (Crucial pour Web/DDoS)
#         elif "PROTOCOL" in col: p = "pr_"    # Protocol (TCP/UDP)
#         elif "DURATION" in col: p = "dur_"   # Duration (Crucial pour BruteForce vs Normal)
#         elif "TOT" in col and "PKTS" in col: p = "cnt_" # Count Packets (Crucial pour DDoS)
#         elif "BYTES" in col: p = "by_"       # Bytes (Crucial pour Exfiltration/Infiltration)
#         elif "FLAGS" in col: p = "fl_"       # Flags (SYN/FIN/RST - Crucial pour Scan)
#         elif "SIZE" in col: p = "sz_"        # Packet Size
#         else: p = "v_"                       # Value (Autre)
#         col_prefixes.append(p)

#     # Conversion optimisée (Vectorisée)
#     # On crée une liste de listes où chaque valeur a son préfixe
#     data_values = df[features].astype(str).values
#     targets = df['LABEL'].astype(str).tolist()
    
#     entries = []
    
#     # On itère sur les lignes (C'est la partie la plus longue, mais nécessaire)
#     for row, target in zip(data_values, targets):
#         # On colle le préfixe à la valeur : "dp_" + "80" -> "dp_80"
#         # C'est ici que T5 devient "Expert" : Il lit "Destination Port 80" au lieu de juste "80"
#         tokenized_row = [f"{prefix}{val}" for prefix, val in zip(col_prefixes, row)]
        
#         entries.append({
#             "input": "|".join(tokenized_row),
#             "target": target
#         })
        
#     return entries

# def balance_training_data(df):
#     """Équilibrage (Uniquement Train)"""
#     balanced_chunks = []
#     unique_labels = df['LABEL'].unique()
    
#     print(f"   ... Équilibrage : {unique_labels}")
    
#     for label in unique_labels:
#         sub = df[df['LABEL'] == label]
#         count = len(sub)
        
#         if label == "Benign":
#             sub = sub.sample(frac=0.2, random_state=42)
#         elif count < MIN_SAMPLES_TRAIN:
#             sub = resample(sub, replace=True, n_samples=MIN_SAMPLES_TRAIN, random_state=42)
        
#         balanced_chunks.append(sub)
        
#     return pd.concat(balanced_chunks)

# # --- 3. EXÉCUTION ---

# def run_phase_1():
#     all_dfs = []
#     final_features = []
    
#     print("--- 1. Lecture & Préparation Expert ---")
#     for filename in tqdm(ALL_FILES, desc="Processing CSVs"):
#         path = os.path.join(DATA_DIR, filename)
#         if not os.path.exists(path): continue
        
#         try:
#             df = pd.read_csv(path, low_memory=False)
#             df, feats = clean_and_map_df(df) # Applique Log-Norm ici
            
#             if df is not None:
#                 all_dfs.append(df)
#                 if not final_features: final_features = feats
#         except Exception as e:
#             print(f"Erreur {filename}: {e}")

#     if not all_dfs: return

#     full_df = pd.concat(all_dfs, ignore_index=True)
#     print(f"Total Données : {len(full_df)}")
    
#     # 2. SPLIT D'ABORD (Critique)
#     print("--- 2. Split Train/Test ---")
#     train_val_df, test_df = train_test_split(full_df, test_size=0.2, stratify=full_df['LABEL'], random_state=42)
#     train_df, val_df = train_test_split(train_val_df, test_size=0.2, stratify=train_val_df['LABEL'], random_state=42)
    
#     del full_df, train_val_df, all_dfs
    
#     # 3. ÉQUILIBRAGE (Train Seulement)
#     print("--- 3. Équilibrage Expert ---")
#     train_df_balanced = balance_training_data(train_df)
    
#     # 4. SAUVEGARDE
#     print("--- 4. Génération TOON avec Préfixes ---")
#     datasets = [("TRAIN", train_df_balanced, TRAIN_FILE), ("VAL", val_df, VALIDATION_FILE), ("TEST", test_df, TEST_FILE)]
    
#     for name, df, filepath in datasets:
#         entries = create_toon_entries(df, final_features) # Applique les préfixes ici
#         with open(filepath, 'w', encoding='utf-8') as f:
#             toml.dump({"entry": entries}, f)
#         print(f"   ✅ {name} sauvegardé ({len(entries)} entrées)")

# if __name__ == "__main__":
#     run_phase_1()

In [4]:
###############################################################
# PHASE 2: Entraînement T5 "Full Scale" (Avec Affichage)
#
# - Hardware : GPU Check.
# - Scheduler : Cosine.
# - Affichage : Barre de progression + Logs fréquents.
###############################################################


# --- 1. CONFIGURATION ---
MODEL_NAME = 't5-small'

# ⚡ TEST RAPIDE : 0.01 = 1% des données. Mettez 1.0 pour le vrai entraînement.
SAMPLE_RATE = 0.01 

DATA_DIR = "/kaggle/input/toon24"  
OUTPUT_DIR = "/kaggle/working/T5_Model_P100/"

# Vérification GPU
if torch.cuda.is_available():
    print(f"✅ GPU DÉTECTÉ : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ATTENTION : TRAIN SUR CPU (LENT)")

# --- 2. CHARGEMENT (CORRIGÉ) ---
# Ajout de l'argument 'sample_rate' dans la définition
def load_dataset_toon(path, sample_rate=1.0):
    print(f"Chargement {path}...")
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = toml.load(f)
    except FileNotFoundError:
        print(f"❌ ERREUR: Fichier introuvable : {path}")
        return None
    
    ds = Dataset.from_list(data['entry']).rename_column("input", "input_text").rename_column("target", "target_text")

    # --- LOGIQUE D'ÉCHANTILLONNAGE ---
    if sample_rate < 1.0:
        num_samples = int(len(ds) * sample_rate)
        # On garde au moins 10 lignes pour éviter les crashs
        num_samples = max(10, num_samples) 
        ds = ds.shuffle(seed=42).select(range(num_samples))
        print(f"   ⚠️ MODE TEST : Dataset réduit à {len(ds)} lignes ({sample_rate*100}%)")
    else:
        print(f"   -> Chargement complet ({len(ds)} lignes)")
    # -----------------------------------

    return ds

# # --- 2. CHARGEMENT ---

# Utilisation de vos noms de fichiers spécifiques
print("--- Préparation des Données ---")
# Maintenant l'appel fonctionne car la fonction accepte l'argument
train_dataset = load_dataset_toon(os.path.join(DATA_DIR, 'train_full (1).toon'), sample_rate=SAMPLE_RATE)
val_dataset = load_dataset_toon(os.path.join(DATA_DIR, 'validation_full (1).toon'), sample_rate=SAMPLE_RATE)

if train_dataset is None or val_dataset is None:
    sys.exit("Arrêt : Fichiers manquants.")
 

✅ GPU DÉTECTÉ : Tesla P100-PCIE-16GB
--- Préparation des Données ---
Chargement /kaggle/input/toon24/train_full (1).toon...
   ⚠️ MODE TEST : Dataset réduit à 26435 lignes (1.0%)
Chargement /kaggle/input/toon24/validation_full (1).toon...
   ⚠️ MODE TEST : Dataset réduit à 15400 lignes (1.0%)


In [5]:

# --- 3. TOKENIZATION (MODE CPU DOUX) ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_IN = 256
MAX_OUT = 16

def process(examples):
    inputs = tokenizer(examples["input_text"], max_length=MAX_IN, truncation=True, padding="max_length")
    targets = tokenizer(examples["target_text"], max_length=MAX_OUT, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

print("⏳ Tokenisation en cours")

# On ajoute batch_size=1000 pour traiter par petits paquets
train_tokenized = train_dataset.map(
    process, 
    batched=True, 
    batch_size=1000, 
    remove_columns=["input_text", "target_text"]
)

val_tokenized = val_dataset.map(
    process, 
    batched=True, 
    batch_size=1000, 
    remove_columns=["input_text", "target_text"]
)

print(" Tokenisation terminée ")


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

⏳ Tokenisation en cours


Map:   0%|          | 0/26435 [00:00<?, ? examples/s]

Map:   0%|          | 0/15400 [00:00<?, ? examples/s]

 Tokenisation terminée 


In [6]:

# --- 4. MODÈLE & CONFIGURATION P100 ---
print("Configuration du Dropout...")
config = AutoConfig.from_pretrained(MODEL_NAME)

config.dropout_rate = 0.1 

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, config=config)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Configuration P100 (Pascal Architecture)
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    
    # --- Optimisation P100 ---
    fp16=True,                       # ✅ OUI : P100 supporte très bien FP16
    bf16=False,                      # ❌ NON : P100 ne supporte PAS BF16
    tf32=False,                      # ❌ NON : P100 ne supporte PAS TF32
    
    # --- Gestion Mémoire ---
    per_device_train_batch_size=64,  # 32 passe généralement sur 16GB VRAM pour T5-Small
    gradient_accumulation_steps=1,   # 32 * 2 = Batch effectif de 64
    per_device_eval_batch_size=64,
    dataloader_num_workers=2,
    
    # --- Stratégie d'Entraînement ---
    learning_rate=3e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # --- Logs & Save ---
    logging_strategy="steps",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    disable_tqdm=False,              # Barre de progression active
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("--- Lancement du Training sur P100 ---")
trainer.train()

final_path = os.path.join(OUTPUT_DIR, "Final")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"✅ Modèle sauvegardé : {final_path}")

Configuration du Dropout...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipykernel_20/1683185249.py:45: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


--- Lancement du Training sur P100 ---


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
1000,0.008400,0.007497


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

✅ Modèle sauvegardé : /kaggle/working/T5_Model_P100/Final


In [7]:
#trainer.train(resume_from_checkpoint=True)

In [8]:
# ###############################################################
# # PHASE 3: Évaluation Finale (Multi-Classe + Binaire)
# ###############################################################

# import toml
# import torch
# import seaborn as sns
# import matplotlib.pyplot as plt
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# from tqdm import tqdm
# import numpy as np

# # --- CONFIGURATION ---
# # Assurez-vous que ces chemins correspondent à vos fichiers Kaggle
# MODEL_PATH = "/kaggle/working/T5_Model_P100/Final"
# TEST_FILE = "/kaggle/input/toon24/test_full (1).toon" 

# # IMPORTANT : Doit être identique à la PHASE 2 (256 ou 512)
# MAX_LEN = 256 
# BATCH_SIZE = 64
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # --- 1. CHARGEMENT ---
# print(f"Chargement du modèle depuis {MODEL_PATH}...")
# try:
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
#     model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
#     model.eval()
# except Exception as e:
#     print(f"Erreur de chargement modèle : {e}")
#     exit()

# # --- 2. LECTURE DONNÉES ---
# print(f"Lecture du fichier test : {TEST_FILE}...")
# with open(TEST_FILE, 'r') as f:
#     data = toml.load(f)

# # Extraction des listes
# inputs = [d['input'] for d in data['entry']]
# targets = [d['target'] for d in data['entry']]

# # --- 3. INFÉRENCE (PRÉDICTIONS) ---
# preds = []
# print(f"Démarrage de l'évaluation sur {len(inputs)} échantillons...")

# # Boucle par batch (plus rapide que un par un)
# for i in tqdm(range(0, len(inputs), BATCH_SIZE)):
#     batch_input = inputs[i : i+BATCH_SIZE]
    
#     # Tokenization
#     tok = tokenizer(
#         batch_input, 
#         max_length=MAX_LEN, 
#         padding="max_length", 
#         truncation=True, 
#         return_tensors="pt"
#     ).to(DEVICE)

#     with torch.no_grad():
#         # Generation (Inference)
#         gen = model.generate(tok["input_ids"], max_length=16)

#     # Décodage (Token IDs -> Texte)
#     decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
#     preds.extend([p.strip() for p in decoded])

# # --- 4. RAPPORT DÉTAILLÉ (MULTI-CLASSES) ---
# print("\n" + "="*50)
# print("RAPPORT 1 : CLASSIFICATION DÉTAILLÉE (Type d'Attaque)")
# print("="*50)
# print(classification_report(targets, preds, zero_division=0))

# # Matrice de Confusion Détaillée
# plt.figure(figsize=(10, 8))
# labels = sorted(list(set(targets)))
# cm = confusion_matrix(targets, preds, labels=labels)
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
# plt.title("Matrice de Confusion : Types d'Attaques")
# plt.ylabel("Réel")
# plt.xlabel("Prédit")
# plt.show()

# # --- 5. RAPPORT BINAIRE (BENIGN vs ATTACK) ---
# print("\n" + "="*50)
# print("RAPPORT 2 : DÉTECTION BINAIRE (Benign vs Attack)")
# print("="*50)

# # Conversion logique : Tout ce qui n'est pas "Benign" devient "Attack"
# def to_binary(label_list):
#     return ["Benign" if x == "Benign" else "Attack" for x in label_list]

# binary_targets = to_binary(targets)
# binary_preds = to_binary(preds)

# # Calcul des métriques
# print(classification_report(binary_targets, binary_preds, zero_division=0))

# # Matrice de Confusion Binaire
# plt.figure(figsize=(6, 5))
# bin_labels = ["Benign", "Attack"]
# cm_bin = confusion_matrix(binary_targets, binary_preds, labels=bin_labels)
# sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Reds', xticklabels=bin_labels, yticklabels=bin_labels)
# plt.title("Matrice de Confusion Binaire (Intrusion Detection)")
# plt.ylabel("Réel")
# plt.xlabel("Prédit")
# plt.show()